In [116]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [117]:
%pip install numpy
%pip install pandas
%pip install pytest
%pip install jinja2

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrad

### Solar Micro-Grid Dispatch Planner

In [118]:
from src.microgrid import MicroGrid

micro_grid = MicroGrid()
determinant = micro_grid.determinant()
condition = micro_grid.condition()

print(f"Determinant is: {determinant}", f"Condition: {condition}")

Determinant is: -4.999999999999999 Condition: 5.82842712474619


### Explain determinant and condition
**Determinant being -4.99999 or 5.** This means the co-efficients are meaningful and this usable to solve the system, if it was 0, it would be the system's equation are not solveable  
**Condtion being 5.8284...** A condition number of about 5.8284 means a 1% error in D1 or D2 can cause at most about a 5.8284% error in the values of x or y. That's small, so the system is well-conditioned. A large condition number , say more that 100, in 1000s or more, would cause significant measurement errors

### Interactive Input

In [119]:
from src.validate import Validate

try:
    validate = Validate()
    
    d1_interactive = input('Provide daytime load (KWh): ')
    validate.check_energy_input(d1_interactive)

    d2_interactive = input('Provide critical-equipment load (kWh):')
    validate.check_energy_input(d2_interactive)
    
    #solve
    result = micro_grid.solve(float(d1_interactive),float(d2_interactive))
    
    print("Results: ",result)
except Exception as e:
    print(e)    



Results:  [ 1.6 -0.4]


### Generate CSV file


In [120]:
from src.csv_generator import CsvGenerator
from pathlib import Path
import csv

csv_file_path = './files/solar_data.csv'
csv_generator = CsvGenerator()

#generate file
csv_generator.solar_grid_csv(csv_file_path,30,36)

#check if csv file exists
if Path(csv_file_path).exists():
    print(f"File {csv_file_path} generate successfully!")


File ./files/solar_data.csv generate successfully!


### Read csv file and solve for all 30 days

**Benchmarking, solve in Loop vs single vectorized cell**

In [121]:
import pandas
import timeit
import numpy
import jinja2

def solve_with_loop(d_data):
    results = []
    #processing with loop
    for l_d1, l_d2 in d_data:
        results.append(micro_grid.solve(l_d1, l_d2))
    return results            

def solve_with_single_v_cell(d_data):
    #solve with single vectorized cell
    return numpy.linalg.solve(micro_grid.co_efficients, d_data.T).T
     

#only run bench marking n_runs number of times
n_runs = 5000

# check if file exists
if Path(csv_file_path).exists():
    with open(csv_file_path, "r") as file:
        rows = list(csv.DictReader(file))
        d_data = numpy.array([[float(r["d1"]), float(r["d2"])] for r in rows])
            
    #solve with loop
    time_taken_loop = timeit.timeit(lambda: solve_with_loop(d_data), number = n_runs)
    
    #solve with single vectorized cell
    time_taken_single_v_cell = timeit.timeit(lambda: solve_with_single_v_cell(d_data),  number = n_runs)
    
    #bench mark results
    benchmark = pandas.DataFrame({
                    "method": ["loop", "vectorized"],
                    "Avg time (ms) per run": [(time_taken_loop * 1e3)/n_runs, (time_taken_single_v_cell * 1e3)/n_runs], #need to get time per run
                    "Avg speed-up": [1.0, time_taken_loop / time_taken_single_v_cell],
                })      
     
    display(benchmark.set_index("method").round(3).style.set_caption(f"Benchmark over {n_runs:,} runs")
    )


    # df = pandas.DataFrame(results,columns=["day", "d1", "d2", "x", "y"])
    # df

else:
     print(f"{csv_file_path} does not exist!")    

,Avg time (ms) per run,Avg speed-up
method,,
loop,0.191000,1.000000
vectorized,0.006000,32.674000


**The vectorized processing is almost 30-times faster than using loop**

In [ ]:
from scipy.optimize import nnls

#generate bad data
bad_csv_file_path = './files/solar_bad_data.csv'
csv_generator_2 = CsvGenerator()
csv_generator_2.solar_grid_csv(bad_csv_file_path,30,36, False)

#check if csv file with bad data exists
if Path(bad_csv_file_path).exists():
    with open(bad_csv_file_path, "r") as file:
        bad_data_rows = list(csv.DictReader(file))

    report = []
    for r in bad_data_rows:
        d1, d2 = float(r["d1"]), float(r["d2"])

        #negative demand is bad data, nothing to solve
        if d1 < 0 or d2 < 0:
            report.append([int(r["day"]), d1, d2, None, None, None, None, None, None, "invalid input"])
            continue

        x, y = micro_grid.solve(d1, d2)
        if x >= -1e-9 and y >= -1e-9:
            fx, fy, status = x, y, "ok"
        else:
            (fx, fy), _ = nnls(micro_grid.co_efficients, [d1, d2])
            status = "infeasible (nnls)"
            cx, cy = max(x, 0), max(y, 0)
            clip_error = ((3*cx + 2*cy - d1)**2 + (4*cx + cy - d2)**2) ** 0.5
        report.append([int(r["day"]), d1, d2, x, y, fx, fy, 3*fx + 2*fy - d1, 4*fx + fy - d2, status])

    report = pandas.DataFrame(report, columns=["day", "d1", "d2", "x_exact", "y_exact", "x", "y", "d1_gap", "d2_gap", "status"]).set_index("day")

    print(report["status"].value_counts())
    display(report[report["status"] != "ok"].round(3))
else:
    print(f"{bad_csv_file_path} does not exist!")

status
ok                   24
invalid input         3
infeasible (nnls)     3
Name: count, dtype: int64


,d1,d2,x_exact,y_exact,x,y,d1_gap,d2_gap,status
day,,,,,,,,,
5,-94.95,73.23,NaN,NaN,NaN,NaN,NaN,NaN,invalid input
10,107.79,161.68,43.114,-10.776,38.804,0.000,8.621,-6.466,infeasible (nnls)
15,93.37,-71.31,NaN,NaN,NaN,NaN,NaN,NaN,invalid input
20,88.59,35.43,-3.546,49.614,5.910,25.974,-18.912,14.184,infeasible (nnls)
25,-94.70,65.85,NaN,NaN,NaN,NaN,NaN,NaN,invalid input
30,93.40,140.10,37.360,-9.340,33.624,0.000,7.472,-5.604,infeasible (nnls)
